# Final Integrated Challenge


## Scenario

You are building a model that predicts whether a person's annual income exceeds **$50K**, using the **Adult Census Income** dataset from OpenML:

```python
from sklearn.datasets import fetch_openml

adult = fetch_openml(name="adult", version=2, as_frame=True)

Your task is to build a fully reproducible Scikit-learn training pipeline, track every experiment with
MLflow, and manage the project history with Git — bringing together everything from the last three
sessions into one complete workflow

# Libraries

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

# Data Loading

In [6]:
adult = fetch_openml(name='adult', version=2, as_frame=True)

X = pd.DataFrame(data=adult.data)
y = pd.Series(adult.target, name=adult.target_names[0])

In [8]:
def data_summary(df: pd.DataFrame, target_col: pd.Series) -> pd.DataFrame:
    """Prints basic data diagnostics and returns missing value summary.

    Args:
        df: Input DataFrame to analyze.
        target_col: Name of the target column to show class distribution for.

    Returns:
        DataFrame of columns with missing values, sorted by missing percentage.
    """
    df = df.copy()
    print('Data shape:')
    print(f'Number of rows: {df.shape[0]}, Number of columns: {df.shape[1]}')
    print("="* 100 )
    print("Target value distribution:")
    print(target_col.value_counts(normalize=True))
    print("="* 100 )
    print(f'Number of Duplicated rows: {X.duplicated().sum()}')
    print("="* 100 )
    missing_values = (df.isna().mean() * 100).round(2)
    col_types = df.dtypes
    unique_values = df.nunique()
    summary_df = pd.DataFrame({
        'perc_missing_values' : missing_values,
        'col_type' : col_types,
        'number_unique_values' : unique_values
    })
    return summary_df[summary_df['perc_missing_values'] > 0.0].sort_values(by='perc_missing_values', ascending=False)



data_summary(df=X, target_col=y)

Data shape:
Number of rows: 48842, Number of columns: 14
Target value distribution:
class
<=50K    0.760718
>50K     0.239282
Name: proportion, dtype: float64
Number of Duplicated rows: 57


,perc_missing_values,col_type,number_unique_values
occupation,5.75,category,14
workclass,5.73,category,8
native-country,1.75,category,41


In [9]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   age             48842 non-null  int64   
 1   workclass       46043 non-null  category
 2   fnlwgt          48842 non-null  int64   
 3   education       48842 non-null  category
 4   education-num   48842 non-null  int64   
 5   marital-status  48842 non-null  category
 6   occupation      46033 non-null  category
 7   relationship    48842 non-null  category
 8   race            48842 non-null  category
 9   sex             48842 non-null  category
 10  capital-gain    48842 non-null  int64   
 11  capital-loss    48842 non-null  int64   
 12  hours-per-week  48842 non-null  int64   
 13  native-country  47985 non-null  category
dtypes: category(8), int64(6)
memory usage: 2.6 MB


In [10]:
numerical_columns = X.select_dtypes(include=np.number).columns.tolist()
categorical_columns = X.select_dtypes(exclude=np.number).columns.tolist()

# Data Split

In [11]:
target_class = {'<=50K' : 0, '>50K' : 1}
y = y.map(target_class)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

len(X_train), len(y_train), len(X_test), len(y_test)

(39073, 39073, 9769, 9769)